In [1]:
# Run this cell first. It installs the packages this notebook uses.
%pip install -q pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


# 03. How a midterm changes what a president can do

## The ideas first, in plain words

**Mandate.** A president's claim that voters endorsed their plans. A big
win strengthens that claim. A midterm loss weakens it, because the newest
election result points the other way.

**Veto.** When the president refuses to sign a bill Congress passed. A
regular veto sends the bill back with objections, and Congress can override
it only with two thirds of each chamber. A pocket veto happens when
Congress adjourns and the president simply does not sign the bill within
ten days. Pocket vetoes cannot be overridden.

**Executive order.** A formal instruction from the president to the
executive branch, for example to an agency. It has the force of law, but it
cannot override a statute, it can be challenged in court, and the next
president can reverse it.

**Confirmation.** The Senate must approve the president's nominees for
federal judges, cabinet secretaries, and other senior posts. If the other
party controls the Senate, confirmations slow down or stop.

**Government shutdown.** If Congress does not pass funding laws, federal
agencies must stop non essential work. Shutdowns are rare, and they are
political events, not automatic consequences of divided government.

**Correlation vs causation.** Correlation means two things move together.
Causation means one thing causes the other. A president who loses a
midterm and then signs fewer bills shows a correlation. But wars, the
economy, court vacancies, and the president's own choices also move those
numbers, so the midterm alone does not prove the cause. This chapter shows
the numbers and names the limits honestly.

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

PROC = os.path.join('..', 'data', 'processed')
CHART_DIR = os.path.join('..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)
plt.rcParams['figure.dpi'] = 120
DEM = '#1f77b4'
REP = '#d62728'
GREY = '#7f7f7f'

def savefig(name):
    path = os.path.join(CHART_DIR, name)
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print('saved', path)

In [3]:
vetoes = pd.read_csv(os.path.join(PROC, 'vetoes.csv'))
eos = pd.read_csv(os.path.join(PROC, 'executive_orders.csv'))
judges = pd.read_csv(os.path.join(PROC, 'judicial_confirmations.csv'))
shutdowns = pd.read_csv(os.path.join(PROC, 'shutdowns.csv'))
mid = pd.read_csv(os.path.join(PROC, 'midterm_history.csv'))
print('Vetoes:'); print(vetoes.to_string(index=False))
print(); print('Executive orders:'); print(eos.to_string(index=False))
print(); print('Judicial confirmations:'); print(judges.to_string(index=False))
print(); print('Shutdowns:'); print(shutdowns.to_string(index=False))

Vetoes:
                president  term_start  term_end  total_vetoes  regular_vetoes  pocket_vetoes  overrides
             Jimmy Carter        1977      1981            31              13             18          2
            Ronald Reagan        1981      1989            78              39             39          9
        George H. W. Bush        1989      1993            44              29             15          1
             Bill Clinton        1993      2001            37              36              1          2
           George W. Bush        2001      2009            12              12              0          4
             Barack Obama        2009      2017            12              12              0          1
Donald Trump (first term)        2017      2021            10              10              0          1
                Joe Biden        2021      2025            13              13              0          0

Executive orders:
                president  term_start

## Vetoes: the president's no

Presidents veto far more when Congress is controlled by the other party,
because Congress keeps sending them bills they oppose. Ronald Reagan
vetoed 78 bills, the most in this period, while facing a Democratic House
for all eight years. George H. W. Bush vetoed 44, also facing a Democratic
Congress. Bill Clinton vetoed 37, most of them after Republicans took
Congress in 1994. By contrast, George W. Bush and Barack Obama vetoed only
12 each, even though both faced opposition Congresses for part of their
terms. Vetoes are a choice, not an automatic meter, so the pattern is
suggestive, not proof.

In [4]:
plt.figure(figsize=(10, 5))
plt.bar(vetoes['president'], vetoes['total_vetoes'], color='#9467bd')
plt.ylabel('Total vetoes (regular plus pocket)')
plt.title('Presidential vetoes by president, Carter through Biden')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', alpha=0.3)
savefig('c03_vetoes.png')
print('Most vetoes:', vetoes.loc[vetoes['total_vetoes'].idxmax(), 'president'],
      vetoes['total_vetoes'].max())
print('Fewest vetoes among completed terms:', vetoes.loc[vetoes['total_vetoes'].idxmin(), 'president'],
      vetoes['total_vetoes'].min())

saved ../charts/c03_vetoes.png
Most vetoes: Ronald Reagan 78
Fewest vetoes among completed terms: Donald Trump (first term) 10


## Executive orders: what a president can do alone

When Congress will not pass the president's program, the president can
still act through executive orders. Jimmy Carter issued 320 in four years,
the highest rate in this period at 80 per year. Ronald Reagan issued 381
over eight years. Donald Trump's first term ran at 55 per year, above the
recent average. But executive orders are fragile: they can be reversed by
the next president, blocked by courts, and they cannot spend money
Congress has not appropriated. They are a workaround, not a replacement
for legislation.

In [5]:
eos['per_year'] = eos['executive_orders'] / (eos['term_end'] - eos['term_start'])
plt.figure(figsize=(10, 5))
plt.bar(eos['president'], eos['executive_orders'], color='#2ca02c')
plt.ylabel('Executive orders in term')
plt.title('Executive orders by president, Carter through Biden')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', alpha=0.3)
savefig('c03_executive_orders.png')
print(eos[['president', 'executive_orders', 'per_year']].round(1).to_string(index=False))

saved ../charts/c03_executive_orders.png
                president  executive_orders  per_year
             Jimmy Carter               320      80.0
            Ronald Reagan               381      47.6
        George H. W. Bush               166      41.5
             Bill Clinton               364      45.5
           George W. Bush               291      36.4
             Barack Obama               276      34.5
Donald Trump (first term)               220      55.0
                Joe Biden               162      40.5


## Confirmations: the Senate's choke point

Federal judges serve for life, so confirmations are one of the longest
lasting things a president does. The Senate must confirm every one. When
the president's party loses the Senate, confirmations slow sharply. The
totals below count Article III judges only: Supreme Court, circuit courts,
and district courts. Different sources count differently (for example,
Ballotpedia counts a few more per president because it includes some
specialty courts), so this project uses one list consistently and says so.
Ronald Reagan's 383 and Bill Clinton's 378 lead the period. Joe Biden's
235 and Donald Trump's first term 234 are close, but they were achieved
under very different Senate conditions, which is a reminder that totals
alone do not tell the whole story.

In [6]:
plt.figure(figsize=(10, 5))
plt.bar(judges['president'], judges['total_article_iii'], color=DEM)
plt.ylabel('Article III judges confirmed')
plt.title('Federal judges confirmed by president, Carter through Biden')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', alpha=0.3)
savefig('c03_judges.png')
print(judges[['president', 'supreme_court', 'circuit', 'district', 'total_article_iii']].to_string(index=False))

saved ../charts/c03_judges.png
                president  supreme_court  circuit  district  total_article_iii
             Jimmy Carter              0       56       206                262
            Ronald Reagan              4       83       296                383
        George H. W. Bush              2       42       149                193
             Bill Clinton              2       66       310                378
           George W. Bush              2       62       263                327
             Barack Obama              2       55       272                329
Donald Trump (first term)              3       54       174                234
                Joe Biden              1       45       189                235


## Shutdowns: the budget fight taken to the limit

There have been six major shutdowns since 1995, totaling 123 days. The
longest was 43 days in October and November 2025, under a Republican
trifecta, which shows that shutdowns are not only a divided government
phenomenon. The 35 day shutdown of December 2018 to January 2019 began
under a Republican trifecta and ended after Democrats took the House in
the 2018 midterm. Shutdowns follow from a failure to agree on funding, and
midterms change who sits at that table.

In [7]:
plt.figure(figsize=(10, 5))
plt.bar(shutdowns['start'] + ' (' + shutdowns['president'] + ')', shutdowns['days'], color=REP)
plt.ylabel('Days')
plt.title('Major federal shutdowns, 1995 to 2025')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', alpha=0.3)
savefig('c03_shutdowns.png')
print('Total shutdown days since 1995:', shutdowns['days'].sum())
print(shutdowns[['start', 'end', 'days', 'president', 'congress_control']].to_string(index=False))

saved ../charts/c03_shutdowns.png
Total shutdown days since 1995: 123
     start        end  days    president                                  congress_control
1995-11-13 1995-11-19     5 Bill Clinton                       Republican House and Senate
1995-12-15 1996-01-06    21 Bill Clinton                       Republican House and Senate
2013-10-01 2013-10-16    16 Barack Obama               Republican House, Democratic Senate
2018-01-20 2018-01-22     3 Donald Trump                       Republican House and Senate
2018-12-22 2019-01-25    35 Donald Trump Republican Senate, Democratic House from Jan 2019
2025-10-01 2025-11-12    43 Donald Trump                       Republican House and Senate


## The mechanisms, in plain words

A midterm changes a president's room to maneuver through four channels.

1. **Laws.** The House and Senate write the laws. If the other party takes
   a chamber, the president's bills stall there. The president can still
   veto the other party's bills, but vetoing is defense, not a program.
2. **Money.** Every spending bill must pass both chambers. A hostile
   chamber can refuse funding for the president's priorities, and in the
   extreme case the government shuts down.
3. **People.** The Senate confirms judges, cabinet members, and
   ambassadors. Losing the Senate means vacancies stay open longer and
   nominees must be more acceptable to the other party.
4. **Oversight.** The party that controls a chamber runs its committees,
   sets hearing agendas, and can investigate the administration. That
   changes the daily political weather even when no law changes.

## What was cut, and why

**Bills enacted per Congress.** This chapter planned to chart public laws
enacted per Congress as the cleanest measure of legislative output. No
clean, verified table of public laws per Congress was secured during the
build, and the project does not guess at one. The chapter therefore uses
vetoes, executive orders, confirmations, and shutdowns, which are all
verified. A future update can add public laws from the Library of
Congress or govinfo once a per Congress table is verified.

**Trump second term indicators.** Vetoes, executive orders, and
confirmations for the second Trump term are in progress and are not
charted as final totals. The veto table notes 2 vetoes so far in the
119th Congress, and executive orders are provisionally about 130 or more
through May 2026. Those numbers will change and are not used in the
charts.

## What to take into the next chapter

A midterm does not remove a president, but it usually removes the
president's ability to pass new laws on their own terms, and it can remove
control of confirmations and the budget. That is the frame for November
2026: the question is not only who wins, but which of these four channels
changes hands.